# Notebook 10: Evaluation Metrics Deep Dive

## Why This Matters

Choosing the wrong evaluation metric is one of the most common and consequential mistakes in applied ML. A model optimized for accuracy on a fraud detection problem will learn to never flag fraud. A model optimized for RMSE on a sales forecast will be dominated by high-value outliers. Every ML interview includes questions about which metric to use and why — and the answer is never "accuracy" or "RMSE" without conditions. This notebook builds deep intuition for classification metrics (the confusion matrix zoo, ROC vs. PR), regression metrics (MAE vs. RMSE vs. MAPE), and cross-validation strategies — including the subtle but critical differences between K-fold, stratified K-fold, and time-series split.

## 1. The Confusion Matrix and Derived Metrics

All binary classification metrics derive from the confusion matrix:

```
                 Predicted +    Predicted -
Actual +      True Positive  False Negative   (row: all positives)
Actual -      False Positive True Negative    (row: all negatives)
```

Key derived metrics and their interpretations:

| Metric | Formula | Interpretation | Null value |
|--------|---------|----------------|------------|
| Accuracy | (TP+TN)/(P+N) | Overall correctness | Majority class % |
| Precision | TP/(TP+FP) | When we say positive, how often right? | Minority class % |
| Recall / TPR | TP/(TP+FN) | Of all positives, how many found? | 0 (predict all negative) or 1 (predict all positive) |
| Specificity / TNR | TN/(TN+FP) | Of all negatives, how many correctly rejected? | — |
| FPR | FP/(FP+TN) | Of all negatives, how many falsely labeled positive? | — |
| F1 | 2PR/(P+R) | Harmonic mean of precision and recall | — |
| MCC | — | Balanced metric even with severe imbalance; range [-1,1] | — |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    TimeSeriesSplit, cross_val_score, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    matthews_corrcoef,
    mean_absolute_error, mean_squared_error,
    mean_absolute_percentage_error, r2_score
)

# Dataset
X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=8,
    weights=[0.85, 0.15], random_state=42
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

lr = LogisticRegression(class_weight='balanced', max_iter=500).fit(X_tr_s, y_tr)
y_pred = lr.predict(X_te_s)
y_prob = lr.predict_proba(X_te_s)[:, 1]

cm = confusion_matrix(y_te, y_pred)
TP, FN, FP, TN = cm[1,1], cm[1,0], cm[0,1], cm[0,0]
print("Confusion matrix:")
print(cm)
print(f"\nTP={TP}  FN={FN}  FP={FP}  TN={TN}")
print(f"Precision:  {TP/(TP+FP):.4f}   (Of predicted positives, how many true?)")
print(f"Recall:     {TP/(TP+FN):.4f}   (Of all positives, how many found?)")
print(f"Specificity:{TN/(TN+FP):.4f}   (Of all negatives, how many correctly rejected?)")
print(f"F1:         {f1_score(y_te, y_pred):.4f}")
print(f"MCC:        {matthews_corrcoef(y_te, y_pred):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay(cm, display_labels=['Negative', 'Positive']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix')

ConfusionMatrixDisplay(
    confusion_matrix(y_te, y_pred, normalize='true'),
    display_labels=['Negative', 'Positive']
).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Normalized Confusion Matrix (row = actual)')
plt.tight_layout()
plt.show()

## 2. ROC Curve vs. Precision-Recall Curve

Both curves show model performance across all thresholds, but they measure different things:

**ROC curve** (Receiver Operating Characteristic):
- X-axis: FPR = FP/(FP+TN) — false alarm rate
- Y-axis: TPR = TP/(TP+FN) — recall
- Area under ROC (AUC-ROC) = probability that the model ranks a random positive higher than a random negative
- Limitation: TN appears in the denominator of FPR. With many true negatives (imbalanced), a model can have low FPR while making many false alarms in absolute terms. This **makes ROC optimistic for imbalanced problems**.

**Precision-Recall curve**:
- X-axis: Recall
- Y-axis: Precision = TP/(TP+FP) — what fraction of positives predictions are correct
- TN does not appear → truly focuses on the minority class performance
- PR-AUC is the right metric for: fraud detection, anomaly detection, any heavily imbalanced problem

Rule of thumb: **use AUC-ROC when classes are roughly balanced; use PR-AUC when the positive class is rare.**

In [ ]:
# Compare ROC and PR curves for balanced vs. imbalanced
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row, (weight, label) in enumerate([(0.5, 'Balanced (50/50)'), (0.97, 'Imbalanced (97/3)')]):
    X_t, y_t = make_classification(
        n_samples=3000, n_features=10, n_informative=5,
        weights=[weight, 1-weight], random_state=42
    )
    X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_t, y_t, test_size=0.3, random_state=42, stratify=y_t)
    sc2 = StandardScaler()
    m2 = LogisticRegression(class_weight='balanced', max_iter=500).fit(sc2.fit_transform(X_tr2), y_tr2)
    y_prob2 = m2.predict_proba(sc2.transform(X_te2))[:, 1]
    
    fpr, tpr, _ = roc_curve(y_te2, y_prob2)
    prec, rec, _ = precision_recall_curve(y_te2, y_prob2)
    auc_roc = roc_auc_score(y_te2, y_prob2)
    auc_pr  = average_precision_score(y_te2, y_prob2)
    
    axes[row][0].plot(fpr, tpr, 'steelblue', linewidth=2)
    axes[row][0].plot([0,1],[0,1],'k--',alpha=0.4)
    axes[row][0].set_title(f'{label}\nROC AUC = {auc_roc:.4f}', fontsize=10)
    axes[row][0].set_xlabel('FPR')
    axes[row][0].set_ylabel('TPR')
    
    baseline_pr = y_te2.mean()
    axes[row][1].plot(rec, prec, 'coral', linewidth=2)
    axes[row][1].axhline(baseline_pr, color='k', linestyle='--', alpha=0.4,
                          label=f'No-skill ({baseline_pr:.3f})')
    axes[row][1].set_title(f'{label}\nPR-AUC = {auc_pr:.4f}', fontsize=10)
    axes[row][1].set_xlabel('Recall')
    axes[row][1].set_ylabel('Precision')
    axes[row][1].legend(fontsize=8)

plt.suptitle('ROC vs. PR Curve: Imbalance Makes ROC Misleadingly High', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Regression Metrics

| Metric | Formula | Interpretation | Use When |
|--------|---------|----------------|----------|
| MAE | mean(|y - ŷ|) | Average absolute error; robust to outliers | Errors are equally important regardless of magnitude |
| MSE | mean((y-ŷ)²) | Penalizes large errors quadratically; differentiable | Need to penalize large errors more; model training |
| RMSE | sqrt(MSE) | Same units as target; easy to interpret | Reporting; when scale matters |
| MAPE | mean(|y-ŷ|/|y|)×100 | % error; scale-invariant | Comparing across different scales; undefined if y=0 |
| R² | 1 - SS_res/SS_tot | Variance explained; 1=perfect, 0=mean predictor | Reporting proportion of variance captured |
| Huber | hybrid MAE/MSE | MAE for small errors, MSE for large → robust | Heavy-tailed error distributions |

In [ ]:
# Show how a single outlier destroys RMSE but not MAE
y_true = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)
y_pred_good = y_true + np.random.RandomState(42).randn(10) * 0.5

for n_outliers in [0, 1, 2, 3]:
    y_pred = y_pred_good.copy()
    y_pred[:n_outliers] += 50  # severe outlier errors
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"Outliers={n_outliers}: MAE={mae:.3f}  RMSE={rmse:.3f}  (RMSE/MAE ratio={rmse/mae:.2f}x)")

print("\nObservation: RMSE explodes with outliers; MAE is stable.")

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor

house = fetch_california_housing()
Xh, yh = house.data, house.target
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh, yh, test_size=0.2, random_state=42)
sc_h = StandardScaler()
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42)
gbr.fit(sc_h.fit_transform(Xh_tr), yh_tr)
yh_pred = gbr.predict(sc_h.transform(Xh_te))

mae  = mean_absolute_error(yh_te, yh_pred)
rmse = np.sqrt(mean_squared_error(yh_te, yh_pred))
mape = mean_absolute_percentage_error(yh_te, yh_pred) * 100
r2   = r2_score(yh_te, yh_pred)

print(f"GBM on California Housing:")
print(f"  MAE:   {mae:.4f} ($100K units)")
print(f"  RMSE:  {rmse:.4f} ($100K units)")
print(f"  MAPE:  {mape:.2f}%")
print(f"  R²:    {r2:.4f} ({r2*100:.1f}% of variance explained)")

# Residual plot
residuals = yh_te - yh_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(yh_pred, residuals, s=5, alpha=0.3, c='steelblue')
axes[0].axhline(0, color='red', linewidth=1)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residual Plot')

axes[1].hist(residuals, bins=50, edgecolor='k', alpha=0.7, color='coral')
axes[1].set_xlabel('Residual')
axes[1].set_title(f'Residual Distribution (skew={pd.Series(residuals).skew():.2f})')

plt.tight_layout()
plt.show()

## 4. Cross-Validation Strategies

A model's training accuracy is not its generalization accuracy. **Cross-validation** estimates the latter by repeatedly holding out different subsets as test sets.

**K-Fold CV**: split data into K folds; train on K-1, test on 1, rotate. Standard choice; use K=5 (fast) or K=10 (lower bias).

**Stratified K-Fold**: maintains class proportions in each fold. Always use for classification, especially with imbalanced data. Without stratification, a fold might have 0 positive examples.

**Leave-One-Out (LOO)**: K=n; maximum data for training; very high variance estimate; only useful for tiny datasets (n < 50).

**TimeSeriesSplit**: for temporal data; the test set must always be in the future relative to training. Never use standard K-Fold on time series — it leaks future information into training folds.

**Group K-Fold**: when rows are not independent — e.g., multiple samples from the same patient or session. Test groups must be disjoint from training groups to prevent leakage.

In [ ]:
# Visualize different CV strategies
n = 20
fig, axes = plt.subplots(4, 1, figsize=(12, 8))
strategies_cv = [
    ('KFold(k=5)', KFold(n_splits=5)),
    ('StratifiedKFold(k=5)', StratifiedKFold(n_splits=5)),
    ('TimeSeriesSplit(n=5)', TimeSeriesSplit(n_splits=5)),
]

X_dummy = np.arange(n).reshape(-1, 1)
y_dummy = np.array([0,1]*10)  # alternating labels

for ax, (name, cv) in zip(axes, strategies_cv):
    for fold_idx, (train, test) in enumerate(cv.split(X_dummy, y_dummy)):
        ax.scatter(train, [fold_idx]*len(train), c='steelblue', s=30, marker='|', linewidths=3)
        ax.scatter(test,  [fold_idx]*len(test),  c='coral',     s=30, marker='|', linewidths=3)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Sample index')
    ax.set_ylabel('Fold')
    ax.set_yticks(range(5))
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='steelblue', label='Train'), Patch(color='coral', label='Test')],
              loc='upper right', fontsize=8)

# Last axis: show gap between KFold and StratifiedKFold on imbalanced data
axes[3].axis('off')
axes[3].text(0.5, 0.5,
    "TimeSeriesSplit: test is ALWAYS in the future\n"
    "StratifiedKFold: preserves class balance in each fold\n"
    "GroupKFold: keeps all rows of a group in same partition",
    ha='center', va='center', fontsize=11, transform=axes[3].transAxes,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Cross-Validation Strategies', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Learning Curves: Diagnosing Bias vs. Variance

A learning curve plots train and validation score as a function of training set size. It directly diagnoses whether you have a bias or variance problem:

- **High bias (underfitting)**: both train and val score are low and plateau at similar values → model is too simple → add complexity or features
- **High variance (overfitting)**: train score is high, val score is low, large gap → model is too complex or needs more data → regularize, reduce features, get more data
- **Good fit**: high train score, val score approaches train score as data grows

In [ ]:
from sklearn.tree import DecisionTreeClassifier

X_lc, y_lc = make_classification(n_samples=3000, n_features=20, n_informative=8, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models = [
    ('High Bias (depth=1)', DecisionTreeClassifier(max_depth=1)),
    ('High Variance (depth=None)', DecisionTreeClassifier(max_depth=None)),
    ('Good Fit (depth=5)', DecisionTreeClassifier(max_depth=5)),
]
train_sizes = np.linspace(0.05, 1.0, 20)

for ax, (name, model) in zip(axes, models):
    tsz, tr_sc, val_sc = learning_curve(
        model, X_lc, y_lc, train_sizes=train_sizes,
        cv=5, scoring='accuracy', n_jobs=-1
    )
    ax.plot(tsz, tr_sc.mean(1),  'steelblue', label='Train', marker='o', markersize=3)
    ax.plot(tsz, val_sc.mean(1), 'coral',     label='Val',   marker='s', markersize=3)
    ax.fill_between(tsz, tr_sc.mean(1)-tr_sc.std(1),  tr_sc.mean(1)+tr_sc.std(1),  alpha=0.1, color='steelblue')
    ax.fill_between(tsz, val_sc.mean(1)-val_sc.std(1), val_sc.mean(1)+val_sc.std(1), alpha=0.1, color='coral')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Accuracy')
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=8)
    ax.set_ylim(0.5, 1.05)

plt.suptitle('Learning Curves: Diagnosing Bias vs. Variance', fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Scenario | Recommended Metric | Why |
|----------|-------------------|-----|
| Balanced binary classification | F1 or AUC-ROC | Single number capturing both precision and recall |
| Imbalanced binary classification | PR-AUC | Focuses on minority class; not inflated by true negatives |
| Ranking quality | AUC-ROC | P(score_pos > score_neg); threshold-independent |
| Regression, no outliers | RMSE or R² | Penalizes large errors; easy to interpret |
| Regression, outlier-prone | MAE or Huber loss | Robust to extreme errors |
| Regression, different scales | MAPE | Scale-invariant; undefined if y=0 |
| Multi-class classification | Macro-F1 (rare classes) or Accuracy (balanced) | Macro treats all classes equally regardless of frequency |

| CV Strategy | Use When |
|-------------|----------|
| KFold | Standard; data is i.i.d.; enough samples per fold |
| StratifiedKFold | Classification; always use to preserve class balance |
| TimeSeriesSplit | Any temporal data; test must always follow training in time |
| GroupKFold | Multiple samples per patient/session/user; groups must not leak |
| LOO | Very small datasets (n < 50); expensive for large n |